## Importing dataset:
We'll be using the 'Predict customer churn' dataset

In [11]:
import pandas as pd 

train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [12]:
#encoding the categorical features
train_df = pd.get_dummies(train_df, drop_first=True)

#splitting the features
x = train_df.iloc[:, :-1].values
y = train_df.iloc[:, -1].values

In [13]:
from sklearn.preprocessing import StandardScaler
import torch

#scaling features
scaler = StandardScaler()
x = scaler.fit_transform(x)

#converting to tensors
x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

In [14]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(x,y)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

## Classifier:

In [15]:
#setting the device to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [16]:
import torch.nn as nn 

class BinaryClassifier(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [17]:
model = BinaryClassifier(input_size=x.shape[1])

In [18]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## Training loop:

In [19]:
model = model.to(device)

In [20]:
for epoch in range(20):
    for x_batch, y_batch in loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        
        #forward pass
        y_pred = model(x_batch)
        loss = loss_fn(y_pred, y_batch)

        #backward pass
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    if epoch % 5 == 0:
        print("Epoch: ", epoch, "Loss: ", loss.item())

Epoch:  0 Loss:  0.31734660267829895
Epoch:  5 Loss:  0.26464182138442993
Epoch:  10 Loss:  0.5302119851112366
Epoch:  15 Loss:  0.3367129862308502


In [22]:
#calculating accuracy:

x = x.to(device)
y = y.to(device)

with torch.no_grad():
    y_pred = model(x)
    y_pred_class = (y_pred > 0.5).float()
    accuracy = (y_pred_class == y).float().mean()
    print("Accuracy: ", accuracy.item())

Accuracy:  0.8571241497993469
